# Advanced NumPy
(follow-up to Numpy_basics.ipynb)

In [ ]:

#* import numpy library
import numpy as np


## 1. Conditional / vectorized logic

In [ ]:

#? np.where() - vectorized if-else 
arr = np.arange(1,11)
print(arr)

result = np.where(arr % 2 == 0, 'even', 'odd')
print(result)

#todo return index that satisfy condition
idx = np.where(arr > 5)
print(idx)

In [ ]:

#* np.select() : vectorize if/elif/else
arr = np.append(arr, [11, 12])
print(arr)
conditions = [arr < 4, (arr >= 4) & (arr < 8), (arr >= 8) & (arr < 11)]
label = ['low', 'mid', 'high']

res = np.select(conditions, label, default='unknown')
print(res)

In [ ]:

#todo np.piecewise : apply different functions to diff. ranges
arr2 = np.arange(-6, 12+1)
print(arr2)

piecewise = np.piecewise(arr2, [arr2 < 0, (arr2 >= 0) & (arr2 < 10) , arr2 >=10], [lambda x: x*(-1), lambda x: 2*x, lambda x: x/2])
print(piecewise)

In [ ]:

#! np.vectorize() : slower than np.where(), np.select()
def custom_func(x):
    return x**2 if x % 2 == 0 else x**3

vec_fn = np.vectorize(custom_func)
arr3 = np.arange(1, 8)
print(vec_fn(arr3))

In [ ]:

#^ np.apply_along_axis() : apply function on single axis (1D)
matrix = np.arange(0, 16).reshape(4, 4)
print(matrix)

def row_sum(row):
    return row.sum()

print(f'\nrow wise sum : {np.apply_along_axis(row_sum, axis=1, arr=matrix)}')

## 2. Broadcasting (properly)

In [ ]:

#* broadcasting rules:
#  - compare shapes from the RIGHT (trailing dims)
#  - dims are compatible if equal, OR one of them is 1
#  - missing dims are treated as 1

a = np.ones((3,4))
b = np.ones((4,))
print((a+b).shape)
print('\n', a)
print('\n', b)

c = np.ones((3, 1))
d = np.ones((1, 5))
print('\n', (c+d).shape)

In [ ]:

#& np.newaxis / np.expand_dims / np.squeeze

vec = np.array([1 , 2, 3])
print('orignal shape : ', vec.shape)

col = vec[:, np.newaxis] #~ (3,) -> (3, 1)
print('colm shape : ', col.shape)

row = np.expand_dims(vec, axis=0) #^ (1, 3)
print('rows shape : ', row.shape)

#? remove shape to 1 dim
squeezed = row.squeeze()
print('squeezed : ', squeezed.shape)

In [ ]:

#todo common broadcasting failure + how to debug it
a = np.ones((3, 4))
b = np.ones((3,))

try:
    a + b
except ValueError as e:
    print('Error : ', e)

#! fix: reshape b to (3,1) so trailing dims align with a's (3,4)
fixed = a + b[:, np.newaxis]
print('\nfixed shape : ', fixed.shape)


In [ ]:

#~ ML-relevant example: normalizing each row (sample) by its own mean/std
data = np.arange(1, 13).reshape(3, 4).astype(float)
print(data)

row_mean = data.mean(axis=1, keepdims=True)   #^ keepdims avoids manual reshape
row_std = data.std(axis=1, keepdims=True)

normalized = (data - row_mean) / row_std
print('\nnormalized : \n', normalized)


## 3. Vectorization & performance

In [ ]:

#* why loops are slow vs vectorized ops
import time

big = np.arange(1_000_000)

#^ loop version
start = time.time()
loop_result = [x**2 for x in big]
loop_time = time.time() - start

#? vectorized version
start = time.time()
vec_result = big ** 2
vec_time = time.time() - start

print(f'loop time  : {loop_time:.4f}s')
print(f'vector time: {vec_time:.4f}s')
print(f'speedup    : {loop_time / vec_time:.1f}x')


In [ ]:

#& ufuncs (universal functions) - operate element-wise, support broadcasting
arr = np.array([1, 4, 9, 16])

print(np.sqrt(arr))     #~ ufunc
print(np.exp(arr))      #~ ufunc
print(np.add(arr, 10))  #~ ufunc, same as arr + 10

#todo ufuncs are implemented in C, that's why they're fast


In [ ]:

#! %timeit demo (run this in a real jupyter kernel to see it live)
arr = np.arange(100000)

#^ uncomment below in an actual notebook cell:
%timeit [x**2 for x in arr]
%timeit arr**2

print('run the commented lines above in a live kernel to compare timings')


## 4. Vector/matrix operations

In [ ]:

#* dot vs matmul vs elementwise multiply
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print('elementwise (*)      : ', a * b)
print('dot product (@)      : ', a @ b)
print('np.dot               : ', np.dot(a, b))
print('np.multiply (=*)     : ', np.multiply(a, b))


In [ ]:

#^ matrix versions
A = np.arange(1, 5).reshape(2, 2)
B = np.arange(5, 9).reshape(2, 2)

print(A)
print('\n', B)
print('\nelementwise (*) :\n', A * B)
print('\nmatmul (@)     :\n', A @ B)
print('\nnp.matmul      :\n', np.matmul(A, B))


In [ ]:

#? cross product (only really meaningful for 3D vectors)
v1 = np.array([1, 0, 0])
v2 = np.array([0, 1, 0])

print('cross product : ', np.cross(v1, v2))


In [ ]:

#~ transpose and axis manipulation
arr3d = np.arange(24).reshape(2, 3, 4)
print('original shape   : ', arr3d.shape)

print('transpose shape  : ', arr3d.T.shape)                 #& reverses all axes
print('swapaxes shape   : ', np.swapaxes(arr3d, 0, 1).shape) #^ swap 2 specific axes
print('moveaxis shape   : ', np.moveaxis(arr3d, 0, -1).shape) #? move one axis to a new position


## 5. Linear algebra (`np.linalg`)

In [ ]:

#* determinant and inverse
A = np.array([[4, 7], [2, 6]], dtype=float)

print('determinant : ', np.linalg.det(A))
print('\ninverse :\n', np.linalg.inv(A))

#^ pseudo-inverse (works even for non-square / singular matrices)
B = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
print('\n', B)
print('\npseudo-inverse :\n', np.linalg.pinv(B))


In [ ]:

#? eigenvalues and eigenvectors - foundational for PCA
A = np.array([[2, 0], [0, 3]], dtype=float)

eigvals, eigvecs = np.linalg.eig(A)
print('eigenvalues  : ', eigvals)
print('eigenvectors :\n', eigvecs)


In [ ]:

#~ solving linear systems: Ax = b
A = np.array([[3, 1], [1, 2]], dtype=float)
b = np.array([9, 8], dtype=float)

x = np.linalg.solve(A, b)
print('solution x : ', x)

#& verify
print('check A @ x == b : ', np.allclose(A @ x, b))


In [ ]:

#todo norms - used in regularization (L1/L2) and distance metrics
v = np.array([3, 4])

print('L2 norm (euclidean) : ', np.linalg.norm(v))        #^ default is L2
print('L1 norm (manhattan) : ', np.linalg.norm(v, ord=1))

#! matrix rank
M = np.array([[1, 2], [2, 4]])   #~ second row is a multiple of first -> rank 1
print('\nmatrix rank : ', np.linalg.matrix_rank(M))


In [ ]:

#* SVD (singular value decomposition) - used in PCA, dimensionality reduction
A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)

U, S, Vt = np.linalg.svd(A)
print('U shape  : ', U.shape)
print('S (singular values) : ', S)
print('Vt shape : ', Vt.shape)

#^ reconstruct original matrix to confirm decomposition is correct
S_full = np.zeros((A.shape[0], A.shape[1]))
np.fill_diagonal(S_full, S)
reconstructed = U @ S_full @ Vt
print('\nreconstruction matches original : ', np.allclose(A, reconstructed))


## 6. Sorting, searching, set operations

In [ ]:

#* sort and argsort
arr = np.array([5, 2, 8, 1, 9])

print('sorted        : ', np.sort(arr))
print('argsort       : ', np.argsort(arr))   #^ indices that would sort the array

#? sort a matrix along an axis
matrix = np.array([[3, 1], [2, 4]])
print('\nsorted rows :\n', np.sort(matrix, axis=1))


In [ ]:

#~ unique values, with counts and first-occurrence indices
arr = np.array([1, 2, 2, 3, 3, 3, 4])

values, index, counts = np.unique(arr, return_index=True, return_counts=True)
print('unique values : ', values)
print('first index   : ', index)
print('counts        : ', counts)


In [ ]:

#& searchsorted - find insertion index to keep array sorted
sorted_arr = np.array([1, 3, 5, 7, 9])
print(np.searchsorted(sorted_arr, 6))   #todo where would 6 go to keep it sorted?


In [ ]:

#! isin - membership check (vectorized 'in')
arr = np.array([1, 2, 3, 4, 5])
targets = np.array([2, 4])

print(np.isin(arr, targets))

#* set operations
a = np.array([1, 2, 3, 4])
b = np.array([3, 4, 5, 6])

print('\nunion        : ', np.union1d(a, b))
print('intersection : ', np.intersect1d(a, b))
print('difference   : ', np.setdiff1d(a, b))


## 7. Handling missing/invalid data

In [ ]:

#* checking for nan/inf
arr = np.array([1, 2, np.nan, 4, np.inf, -np.inf])
print(arr)

print('isnan : ', np.isnan(arr))
print('isinf : ', np.isinf(arr))

#^ replace nan/inf with usable numbers
cleaned = np.nan_to_num(arr, nan=0.0, posinf=999, neginf=-999)
print('\ncleaned : ', cleaned)


In [ ]:

#? nan-safe aggregations - ignore nan instead of propagating it
arr = np.array([1, 2, np.nan, 4, 5])

print('regular mean (-> nan) : ', np.mean(arr))
print('nanmean               : ', np.nanmean(arr))
print('nansum                : ', np.nansum(arr))
print('nanstd                : ', np.nanstd(arr))


In [ ]:

#~ masked arrays - mark specific values as invalid without deleting them
import numpy.ma as ma

arr = np.array([1, 2, -999, 4, 5])   #& -999 used as a 'missing' sentinel
masked = ma.masked_equal(arr, -999)

print(masked)
print('mean ignoring masked value : ', masked.mean())


## 8. Structured/record arrays

In [ ]:

#* structured array with named fields (precursor to pandas DataFrame)
dtype = [('name', 'U10'), ('age', 'i4'), ('salary', 'f8')]

people = np.array([
    ('Amit', 28, 55000.0),
    ('Riya', 34, 72000.0),
    ('Sam', 25, 48000.0)
], dtype=dtype)

print(people)
print('\nnames   : ', people['name'])   #^ access by field name
print('ages    : ', people['age'])
print('avg sal : ', people['salary'].mean())


## 9. Random module (modern `Generator` API)

In [ ]:

#* legacy way (what we used earlier - still works, but not best practice)
np.random.seed(42)
print(np.random.rand(3))

#^ modern recommended way
rng = np.random.default_rng(seed=42)
print('\nuniform [0,1) : ', rng.random(3))


In [ ]:

#? sampling from distributions with the new Generator
rng = np.random.default_rng(seed=1)

print('normal      : ', rng.normal(loc=0, scale=1, size=5))
print('binomial    : ', rng.binomial(n=10, p=0.5, size=5))
print('poisson     : ', rng.poisson(lam=3, size=5))
print('choice      : ', rng.choice([10, 20, 30, 40], size=3, replace=False))


In [ ]:

#~ seeding for reproducibility - same seed always gives same output
rng1 = np.random.default_rng(seed=99)
rng2 = np.random.default_rng(seed=99)

print(rng1.integers(0, 100, 5))
print(rng2.integers(0, 100, 5))   #& identical to rng1's output


## 10. Advanced indexing

In [ ]:

#* fancy indexing - index with an array of indices, not just a slice
arr = np.arange(10, 20)
print(arr)

idx = np.array([1, 3, 5])
print(arr[idx])

#^ works on matrices too, per axis
matrix = np.arange(1, 17).reshape(4, 4)
print('\n', matrix)
print('\nselected rows :\n', matrix[[0, 2]])
print('\nselected colms : \n', matrix[:,[0,3]])

In [ ]:

#? np.take and np.put
arr = np.arange(10, 20)
print(arr)
print(np.take(arr, [0, 2, 4]))   #~ same as fancy indexing, useful for axis-based take

arr2 = np.arange(10, 20)
np.put(arr2, [0, 1], [999, 888])  #& modify specific positions in place
print(arr2)


In [ ]:

#todo np.where() as index-returning (not just conditional replacement)
arr = np.array([10, 25, 3, 47, 8])

idx = np.where(arr > 10)
print('indices where > 10 : ', idx)
print('values             : ', arr[idx])


In [ ]:

#! combining boolean + fancy indexing
matrix = np.arange(1, 26).reshape(5, 5)
print(matrix)

#* select specific rows, then apply a boolean mask on columns
rows = matrix[[0, 2, 4]]
result = rows[:, rows[0] % 2 == 0]
print('\ncombined result :\n', result)


## 11. Array manipulation extras

In [ ]:

#* general concatenate (vstack/hstack are just wrappers around this)
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print('axis=0 :\n', np.concatenate((a, b), axis=0))
print('\naxis=1 :\n', np.concatenate((a, b), axis=1))


In [ ]:

#^ tile and repeat
arr = np.array([1, 2, 3])

print('tile   : ', np.tile(arr, 2))     #~ repeats the whole array
print('repeat : ', np.repeat(arr, 2))   #& repeats each element


In [ ]:

#? pad - add values around an array
arr = np.array([1, 2, 3])

padded = np.pad(arr, pad_width=5, mode='constant', constant_values=(0,7))
print(padded)


In [ ]:

#todo flip and roll
arr = np.arange(1, 6)

print('orignal : ', arr)
print('flip : ', np.flip(arr))
print('roll : ', np.roll(arr, shift=-2))   #! shifts elements circularly


## 12. Einsum (optional, powerful)

In [ ]:

#* einsum - compact way to express tensor operations
A = np.arange(1, 5).reshape(2, 2)
B = np.arange(5, 9).reshape(2, 2)

#^ matrix multiplication via einsum
print('matmul via einsum :\n', np.einsum('ij,jk->ik', A, B))

#? transpose via einsum
print('\ntranspose via einsum :\n', np.einsum('ij->ji', A))

#~ trace (sum of diagonal) via einsum
print('\ntrace via einsum : ', np.einsum('ii->', A))

#& dot product of vectors via einsum
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])
print('\ndot product via einsum : ', np.einsum('i,i->', v1, v2))
